# 8. 提升方法

## 引入概念：GBDT

**GBDT** 全称 **Gradient Boosting Decision Tree**，梯度提升决策树，属于 Boosting 串行集成学习算法，基学习器是 CART 回归树。

**含义**：一棵一棵树依次串行训练，每棵新树去拟合上一轮模型损失函数的负梯度（伪残差），最后把所有树预测值累加起来得到最终结果。

**理解**：第一棵树先粗略预测，第二棵专门修正第一棵的误差，第三棵修正前两棵合起来的误差，不断补坑。


## 1. 引言

回归：随机森林，它是装袋 Bagging 思想：一次性同时训练一大堆比较强、但是不稳定的大树，每棵树自己独立学、互不打扰，最后大家一起投票取平均。（**并行**）

例如：随机森林 = 班里几十个中等水平的同学，同时独立做题，最后答案一起投票，抵消各自的失误。

现在换一条完全相反的思路：如果我手里没有这些中等水平的同学，只有很多很多水平很差的学生 —— 他们只比瞎蒙强一点点，单独一个人做题正确率很低，能不能把这群 "差生" 组合起来，变成一个很厉害的高手？

这个就需要使用：**Boosting，提升方法**。


## 2. 概念

PAC 学习、弱可学习 → 强可学习

很早的时候学界就在争论这个问题：

- **弱学习器（弱模型）**：效果仅仅比随机瞎猜好一点点的模型
- **强学习器（强模型）**：可以做到误差非常小、预测很准的模型

科学家提出猜想：如果我只有弱模型，能不能拼出强模型？**1990 年 Schapire 证明了：可以！**

他最早的思路特别好懂，就是 "三个臭皮匠" 版本：

1. 先训练第一个弱模型 A，它会做错一部分样本；
2. 专门挑 A 做错的那些题目，训练第二个弱模型 B，让 B 专门搞定 A 不会的题；
3. 再训练第三个弱模型 C，专门用来判断：A 和 B 答案不一样的时候，谁说的对；最后三个一起投票，整体准确率就明显高于单独任何一个。

而且这个思路可以不断叠加上去，不断新增弱模型、持续补错，误差可以无限压低。"Boosting（提升）" 这个名字就是这么来的 —— 把弱能力 "提升" 成强能力。

**记住 Boosting 的核心规则：串行排队训练，后面的模型，专门去弥补前面所有模型犯的错误。**

**Boosting 与随机森林对比：**

- 随机森林：并行，树之间互不依赖，可以一起训练；
- Boosting：串行，必须等上一个模型训练完，算出哪里错了，下一个才知道要学什么，不能并行。


## 3. 一个树桩有多弱

提升里最常用的弱模型是**树桩（decision stump）**：只切一刀的决策树。

案例：比较树桩、深度 3 的树和长满的树的准确率。


In [1]:
# 准备数据
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# 定义函数
def make_2d(seed=273, n=800):
    # 固定随机种子，保证每次运行生成一样的数据
    r = np.random.default_rng(seed)
    # 月收入：3千~25千，保留1位小数
    inc = np.round(r.uniform(3, 25, n), 1)
    # 负债收入比，限制在0.01~0.99之间
    debt = np.round(np.clip(r.beta(2.2, 3, n), 0.01, 0.99), 3)
    
    # 真实规则：负债 > 0.22 + 0.022*收入， 或者收入<6千同时负债>0.2 → 属于高风险
    risky = (debt > 0.22 + 0.022 * inc) | ((inc < 6) & (debt > 0.2))
    
    # 随机挑12%样本翻转标签，制造噪声：^ 是异或，满足risky，但抽到噪声样本，标签反过来
    y = (risky ^ (r.random(n) < 0.12)).astype(int)
    
    # 把收入、负债两个特征拼在一起，形成两列特征矩阵
    X = np.c_[inc, debt]
    
    # 划分训练、验证
    return X[:200], y[:200], X[200:], y[200:]

# 生成数据
X2, y2, X2v, y2v = make_2d()

# 计算基准线：全猜多数类的准确率
print("全猜多数类：", round(max(y2v.mean(), 1 - y2v.mean()), 4))

# 训练并对比不同深度的决策树
for name, depth in [('一个树桩', 1), ("深度3的树", 3), ("长满的树", None)]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X2, y2)
    print(f"{name}: 训练准确率 {m.score(X2, y2):.4f}, 验证准确率 {m.score(X2v, y2v):.4f}")

全猜多数类： 0.5983
一个树桩: 训练准确率 0.7250, 验证准确率 0.7083
深度3的树: 训练准确率 0.8250, 验证准确率 0.8117
长满的树: 训练准确率 1.0000, 验证准确率 0.7067


一个树桩的验证准确率 0.708，比全猜多数类（0.598）强一些，可也只强这么一点：它只能沿着一个特征切一刀，斜着走的边界完全画不出来。它**偏差大、方差小**：换几个人重新训练，那一刀大概还在差不多的位置。正好和第 6 讲长满的树相反。随机森林用平均压方差；提升要做的，是把一个个偏差大的树桩，一步步组合成偏差小的模型。


## 4. 如何进行一轮一轮的训练

补充：树桩就是 Boosting（AdaBoost 等提升算法）最经典的弱学习器。

前面：AdaBoost 就是一轮一轮训练树桩（只切一刀的弱分类器），后一个树桩专门去搞定前一个树桩分错的样本。

核心手段：给每个样本带上权重 $w$：

- 权重越大，代表这个样本越重要；
- 训练下一个树桩的时候，优先保证把权重高的样本分对；
- 第一轮：所有样本权重一模一样，大家同等重要；
- 衡量好坏不再是 "分错几个"，而是加权错误 $\varepsilon$ = 所有分错样本的权重加起来。

标签用 +1（违约）、-1（正常），不用 0、1，是为了后面 AdaBoost 公式写起来简洁，这是 AdaBoost 的惯例。

AdaBoost 的回答是样本权重：每个人有一个权重，加起来是 1；找一刀时，看的是加权错误（判错的人的权重之和）。例如：10 个人（两个特征 $x_1$、$x_2$，1 表示违约、−1 表示正常）。

| 样本 | x1 | x2 | y | x1 ≤ 5.5？ | 树桩判断 h | 是否判错 | 权重 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 2 | 8 | +1 | 是 | −1 | 错 | 0.1 |
| 2 | 9 | 8 | +1 | 否 | +1 | 对 | 0.1 |
| 3 | 4 | 4 | −1 | 是 | −1 | 对 | 0.1 |
| 4 | 9 | 1 | +1 | 否 | +1 | 对 | 0.1 |
| 5 | 3 | 6 | −1 | 是 | −1 | 对 | 0.1 |
| 6 | 6 | 6 | +1 | 否 | +1 | 对 | 0.1 |
| 7 | 8 | 5 | −1 | 否 | +1 | 错 | 0.1 |
| 8 | 1 | 7 | −1 | 是 | −1 | 对 | 0.1 |
| 9 | 5 | 5 | −1 | 是 | −1 | 对 | 0.1 |
| 10 | 4 | 9 | −1 | 是 | −1 | 对 | 0.1 |

1 号和 7 号判错，加权错误 `ε = 0.1 + 0.1 = 0.2`，下一轮把这两个人的权重调大。

`best_stump(X, y, w)` 遍历每个特征、以相邻取值的中点为候选阈值，并试两个方向 `s = ±1`；预测 `h = np.where(X[:, j] <= t, s, -s)`，用 `err = w[h != y].sum()` 计算加权错误，返回误差最小的那一刀。

In [2]:
# 目的：一共10个人，2个特征，第一轮所有人权重都是0.1
# 遍历所有可能的“一刀切”，找到【加权错误最小】的那一刀，也就是最优树桩
import numpy as np

# X10：10行2列，10个样本，两列分别是特征x1、x2
X10 = np.array([[2, 8], [9, 8], [4, 4], [9, 1], [3, 6], [6, 6], [8, 5], [1, 7], [5, 5], [4, 9]])
# y10：标签，1代表违约，-1代表正常，AdaBoost经典±1标记
y10 = np.array([1, 1, -1, 1, -1, 1, -1, -1, -1, -1])
# w：权重数组，一共10个样本，初始权重全部等于0.1；所有权重相加总和=1
w = np.full(10, 0.1)

def best_stump(X, y, w):
    """
    遍历每一个特征、每一个阈值、两种判定方向
    返回：加权误差最小的那一刀（最优树桩）
    返回内容：(最小加权错误, 第几号特征, 阈值, 判定方向s, 这一刀对全部样本的预测结果h)
    """
    best = None  # 用来保存目前找到最好的一刀，初始为空
    # 循环：遍历每一列特征 j，j=0是x1，j=1是x2
    for j in range(X.shape[1]):
        # 取出当前特征这一列所有的值，去重，得到该特征所有不同取值
        vals = np.unique(X[:, j])
        # 相邻两个取值中点作为候选阈值，这是决策树找分割点的标准做法
        # 比如取值是 [1,3,4]，中点就是 (1+3)/2=2、(3+4)/2=3.5
        for t in (vals[:-1] + vals[1:]) / 2:
            # s 代表判定方向：两种选择
            # s=1：如果样本特征 ≤阈值 → 判为+1（违约）
            # s=-1：如果样本特征 >阈值 → 判为-1（正常）
            for s in [1, -1]:
                # h：当前这一刀对全部样本的预测结果数组
                # np.where：满足 X[:,j] ≤ t → 输出s；否则输出-s
                h = np.where(X[:, j] <= t, s, -s)
                # 计算加权错误：把【预测h不等于真实标签y】的那些样本的权重加起来
                err = w[h != y].sum()
                # 如果还没有最优，或者当前这一刀误差比已经存的最优更小 → 更新最优
                # 减 1e-12 是浮点数精度容错，避免浮点相等带来的bug
                if best is None or err < best[0] - 1e-12:
                    best = (err, j, t, s, h)
    return best

# 调用函数，拿到最优树桩的全部信息
eps, j, t, s, h = best_stump(X10, y10, w)
# 打印结果，给人看懂：j从0开始，所以特征编号+1
print(f"最好的一刀：特征 x{j + 1} ≤ {t} 判{'违约' if s == 1 else '正常'}，否则判{'正常' if s == 1 else '违约'}")
print("这一刀的判断：", h.tolist())
# 找到所有预测错误的样本下标（从1开始编号给学生看）
wrong_list = [k + 1 for k in range(10) if h[k] != y10[k]]
print(f"判错的人：{wrong_list}   加权错误 ε = {round(eps, 4)}")

最好的一刀：特征 x1 ≤ 5.5 判正常，否则判违约
这一刀的判断： [-1, 1, -1, 1, -1, 1, 1, -1, -1, -1]
判错的人：[1, 7]   加权错误 ε = 0.2


## 5. 发言权 α 和新的权重

AdaBoost：不断训练弱分类器（简单一刀），给每个弱分类器分配发言权 $\alpha$，同时调整样本权重，让下一个分类器重点关注上一轮分错的样本。

例子：一共 10 个样本，这个弱分类器猜对 8 个、猜错 2 个，错误率 $\varepsilon = 2/10 = 0.2$。

这一刀判对了 80%，比瞎猜（50%）强。强多少，就给它多大的发言权：

$$\alpha = \frac{1}{2}\ln\frac{1-\varepsilon}{\varepsilon}$$

$\varepsilon$ 是本轮错误率：

- 分类器很准（$\varepsilon$ 很小）→ $\alpha$ 很大，投票话语权很重；
- 分类器很差（$\varepsilon$ 接近 0.5，和瞎蒙一样）→ $\alpha \approx 0$，几乎没有发言权；
- 分类器比瞎猜还差（$\varepsilon > 0.5$）→ $\alpha$ 是负数，相当于反过来用它的结果。

这个例子里 $\varepsilon = 0.2$，算出 $\alpha = 0.5 \times \ln(4) \approx 0.693$。

**更新样本权重：错的加重、对的减轻**

然后改权重：判错的人乘 $e^{\alpha}$（变重），判对的人乘 $e^{-\alpha}$（变轻），再除以总和、重新凑成 1。用 $y_i \cdot h(x_i)$ 判断对错（判对是 1，判错是 −1），一行就能写完：

$$w \leftarrow w \cdot e^{-\alpha \cdot y_i h(x_i)}$$

- 猜对的时候：$y_i h(x_i) = 1$，乘 $e^{-\alpha}$ → 权重变小
- 猜错的时候：$y_i h(x_i) = -1$，乘 $e^{\alpha}$ → 权重变大

$Z$ 是归一化因子：全部权重求和，除以 $Z$，保证所有权重加起来 = 1（概率分布）。

> AdaBoost 最巧妙的设计：抹平当前分类器的能力，强制下一轮弱学习器去学习剩下的、没搞定的样本。

**说明**：下面的单元格用 `y10_h` 表示 `y * h`，但数组里实际写了 6 个 −1（正好是上一格的预测 `h`），而不是 `y10 * h` 得到的 2 个 −1。因此运行后“判错的人的权重合计”是 0.8571。严格按上面表格（1 号和 7 号判错）计算，`y10 * h = [-1, 1, 1, 1, 1, 1, -1, 1, 1, 1]`，两个判错样本的权重从 0.1 升到 0.25，其余 8 人降到 0.0625，判错权重合计为 0.5。

## 6. 三轮合起来：按发言权投票

前面：找一刀（弱分类器）→ 算发言权 $\alpha$ → 修改样本权重，让错的样本变重要。

重复 "找一刀 → 算 $\alpha$ → 改权重"，三轮以后，把三刀的判断按发言权加起来，看正负；重复多轮，最后把所有弱分类器按发言权加权投票，拼成一个很强的模型！


In [3]:
# 导入numpy，用来做向量、指数、对数计算
import numpy as np
# 1、设置本轮弱分类器的错误率eps --------
eps = 0.2
# 例子：10个样本，错2个，错误率=2/10=0.2
# 2、计算这个弱分类器的发言权 alpha --------
# alpha = 0.5 * ln( (1-eps)/eps )，AdaBoost标准公式
alpha = 0.5 * np.log((1 - eps) / eps)
# 准备标记y10：标记每个样本本轮是猜对还是猜错
# y10 * h = 1 代表猜对；y10 * h = -1 代表猜错
# 这里：10个样本，2个猜错，8个猜对
# 我们写成 y10*h 的结果，2个-1，8个1
y10_h = np.array([-1, 1, -1, 1, -1, 1, 1, -1, -1, -1])
# 4、初始权重：10个样本一开始权重均等，每个0.1
w = np.ones(10) / 10
# 5、权重更新：乘上系数 e^(-alpha * y10_h)，错的乘e^α，对的乘e^(-α)
w_new = w * np.exp(-alpha * y10_h)
# 打印中间结果
print("α =", round(alpha, 4), "  e^α =", round(np.exp(alpha), 4), "  e^(-α) =", round(np.exp(-alpha), 4))
print("乘完指数以后（还没归一化）：", np.round(w_new, 4).tolist(), "  合计", round(w_new.sum(), 4))
# 归一化：除以总和Z，让所有权重之和=1
w_new = w_new / w_new.sum()
# 打印归一之后的权重
print("归一化后：", np.round(w_new, 4).tolist())
# 本轮猜错的那些样本
# y10_h == -1 就是猜错的样本
print("判错的人的权重合计：", round(w_new[y10_h != 1].sum(), 4))


α = 0.6931   e^α = 2.0   e^(-α) = 0.5
乘完指数以后（还没归一化）： [0.2, 0.05, 0.2, 0.05, 0.2, 0.05, 0.05, 0.2, 0.2, 0.2]   合计 1.4
归一化后： [0.1429, 0.0357, 0.1429, 0.0357, 0.1429, 0.0357, 0.0357, 0.1429, 0.1429, 0.1429]
判错的人的权重合计： 0.8571


- $h_m(x)$：第 m 把 "刀"，它的输出只有两种：+1、-1（比如 +1 代表判违约，-1 代表正常）
- $\alpha_m$：这把刀的发言权，准确率越高，$\alpha_m$ 越大，投票分量越重
- $F(x) = \sum \alpha_m h_m(x)$：加权总分，把每一把刀的判断 × 它的发言权，全部加起来
- $\text{sign}(F(x))$：符号函数。总分 > 0 → 最终判定 +1；总分 < 0 → 最终判定 -1

核心思想：每一轮只需要一个 "比瞎猜好一点点" 的弱分类器（$\varepsilon < 0.5$），多轮加权投票，强强联合，最后整体正确率很高。

α 随 ε 的变化：

1. $\varepsilon$ 越小（本轮弱分类器错得越少）→ $\alpha$ 越大，投票话语权越强；
2. $\varepsilon = 0.5$（和抛硬币瞎猜一样）→ $\alpha = 0$，这把刀直接没有发言权，等于不用它；
3. $\varepsilon > 0.5$（这个分类器还不如瞎猜，反向输出才准）→ $\alpha$ 是负数，相当于把它的判断反过来用。

3 轮运行结果：第 1 轮 `x1 ≤ 5.5` 判正常，第 2 轮 `x2 ≤ 7.5` 判正常，第 3 轮又是 `x1 ≤ 5.5` 判正常；`F(x)` 的符号与真实标签相比，10 个里对 8 个。

In [4]:
# 目的：把“找一刀 → 算 α → 改权重”重复 3 轮，再按 α 加权投票
def adaboost(X, y, rounds):
    w = np.full(len(y), 1 / len(y)) # 初始化权重，全部分为 1/N
    model = []                      # 用来存储每一轮的 (alpha, 特征, 阈值, 方向)
    
    for m in range(rounds):
        # 1. 找当前权重下，加权误差最小的树桩
        eps, j, t, s, h = best_stump(X, y, w)
        
        # 2. 计算这个树桩的话语权 alpha
        # eps 越小，alpha 越大，话语权越重
        alpha = 0.5 * np.log((1 - eps) / eps)
        
        # 3. 保存当前这棵树的参数
        model.append((alpha, j, t, s))
        
        # 4. 核心：更新样本权重（错题本机制）
        # 如果预测正确（y*h=1），权重乘 exp(-alpha) < 1，权重变小；
        # 如果预测错误（y*h=-1），权重乘 exp(alpha) > 1，权重变大。
        w = w * np.exp(-alpha * y * h)
        # 归一化，保证权重总和为 1
        w = w / w.sum()
        
    return model

def ada_score(model, X):
    """F(x) = Σ α·h(x)：正的判违约，负的判正常"""
    # 将所有树桩的预测结果（±1）乘以它们的话语权 alpha，然后累加
    return sum(a * np.where(X[:, j] <= t, s, -s) for a, j, t, s in model)

# 调用函数，跑 3 轮
model3 = adaboost(X10, y10, 3)

# 打印每一轮选出的弱分类器及其权重
for m, (a, j, t, s) in enumerate(model3, 1):
    print(f"第 {m} 轮：x{j + 1} ≤ {t} 判{'违约' if s == 1 else '正常'}，α = {a:.4f}")

# 获取最终的集成得分 F
F = ada_score(model3, X10)
print("F(x) =", np.round(F, 3).tolist())
print("sign(F) 和真实结果一样的人数：", int((np.sign(F) == y10).sum()), "/ 10")

第 1 轮：x1 ≤ 5.5 判正常，α = 0.6931
第 2 轮：x2 ≤ 7.5 判正常，α = 0.7332
第 3 轮：x1 ≤ 5.5 判正常，α = 0.4055
F(x) = [-0.365, 1.832, -1.832, 0.365, -1.832, 0.365, 0.365, -1.832, -1.832, -0.365]
sign(F) 和真实结果一样的人数： 8 / 10


## 7. sklearn 中 AdaBoost 使用

前面我们自己手写实现了 AdaBoost，现在：调用 sklearn 库里现成的 AdaBoost 分类器，和咱们手写版本做对照、找差别。

先抓住最重要的知识点：

1. 基础弱分类器：`DecisionTreeClassifier(max_depth=1)`，就是决策树桩（stump）—— 只能切一刀。
2. `n_estimators=3`：一共训练 3 个弱分类器（3 轮）。

### 7.1 案例实现

sklearn 默认使用 SAMME，`estimator_weights_` 里的权重不带手写公式前面的 `1/2`，所以数值是手写 α 的 2 倍；`estimator_errors_` 是每一轮的加权错误，`tree_.feature` 与 `tree_.threshold` 可以查看树桩用了哪个特征、切在哪里。

In [5]:
# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 准备数据
X10 = np.array([[1,7],[2,8],[3,4],[9,1],[3,6],[4,5],[8,5],[1,8],[2,6],[4,9]])
y10 = np.array([1,1,-1,1,-1,-1,1,1,-1,-1])

# 初始化并训练AdaBoost
# estimator：弱分类器，max_depth=1 代表只能切一刀（决策树桩stump）
# n_estimators=3：一共训练3个弱分类器，也就是3轮
# random_state=0：固定随机种子，保证每次运行结果不变
ada3 = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=3,
    random_state=0
).fit(X10, y10)

# 循环遍历每一轮训练出来的弱分类器，从1开始编号
for m, tr in enumerate(ada3.estimators_, 1):
    # estimator_weights_：sklearn保存的每个弱分类器权重（SAMME，不带1/2）
    a = ada3.estimator_weights_[m - 1]
    # estimator_errors_：每一轮弱分类器的加权错误率ε
    eps = ada3.estimator_errors_[m - 1]
    # tr.tree_.feature[0]：这一刀切的是第几个特征；tr.tree_.threshold[0]：分割阈值
    print(f"第{m}轮：x{tr.tree_.feature[0]+1}<={tr.tree_.threshold[0]},加权错误 {eps:.4f},sklearn的权重 {a:.4f},除以2是 {a/2:.4f}")

# 用训练好的AdaBoost对10个样本进行预测
print("预测：", ada3.predict(X10).tolist())

第1轮：x1<=1.5,加权错误 0.3000,sklearn的权重 0.8473,除以2是 0.4236
第2轮：x1<=6.0,加权错误 0.3095,sklearn的权重 0.8023,除以2是 0.4012
第3轮：x2<=6.5,加权错误 0.2586,sklearn的权重 1.0531,除以2是 0.5266
预测： [1, -1, -1, 1, -1, -1, 1, 1, -1, -1]


## 8. 训练的轮数越多越好吗？

前面我们以为：AdaBoost 轮数加得越多，模型就越强、越准。

**核心问题：轮数越多，一定越好吗？**

AdaBoost 对噪声样本非常敏感！因为它会不断提高错分样本的权重；如果那个样本本身就是标错的噪声，权重越拉越高，后面一堆弱分类器专门围着这个噪声去拟合，最后把正常规律破坏掉。

所以轮数不是越大越好，存在一个最优轮数，到这个轮数就该停下。

### 8.1 案例

下面的代码训练 500 轮，用 `staged_predict` 依次查看前 1、2、…、500 个弱分类器组合的预测，对比训练集和验证集的准确率。

In [6]:
# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 定义函数
def make_2d(seed=273, n=800):
    # 固定随机种子，保证每次运行生成一样的数据
    r = np.random.default_rng(seed)
    # 月收入：3千~25千，保留1位小数
    inc = np.round(r.uniform(3, 25, n), 1)
    # 负债收入比，限制在0.01~0.99之间
    debt = np.round(np.clip(r.beta(2.2, 3, n), 0.01, 0.99), 3)
    
    # 真实规则：负债 > 0.22 + 0.022*收入， 或者收入<6千同时负债>0.2 → 属于高风险
    risky = (debt > 0.22 + 0.022 * inc) | ((inc < 6) & (debt > 0.2))
    
    # 随机挑12%样本翻转标签，制造噪声
    y = (risky ^ (r.random(n) < 0.12)).astype(int)
    
    # 把收入、负债两个特征拼在一起，形成两列特征矩阵
    X = np.c_[inc, debt]
    
    # 划分训练、验证
    return X[:200], y[:200], X[200:], y[200:]

# 初始化数据
X2, y2, X2v, y2v = make_2d()

# 训练AdaBoost，一共500轮，弱分类器还是一刀树桩
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=500,
    random_state=0
).fit(X2, y2)

# staged_predict: 依次返回【只用前1个、前2个、……、前500个弱分类器】的预测结果
# tr_acc：训练集准确率数组，长度500
tr_acc = [np.mean(p == y2) for p in ada.staged_predict(X2)]
# va_acc：验证集准确率数组，长度500
va_acc = [np.mean(p == y2v) for p in ada.staged_predict(X2v)]

# 挑选几个有代表性轮数打印结果：1,5,10,50,100,200,500轮
for m in [1, 5, 10, 50, 100, 200, 500]:
    # m-1是数组下标（从0开始）
    print(f"{m:>3}轮：训练{tr_acc[m-1]:.4f}，验证 {va_acc[m-1]:.4f}")

# np.argmax找到验证准确率最大值对应的下标，+1转成轮号
best_round = int(np.argmax(va_acc)) + 1
print(f"验证准确率最高：第 {best_round} 轮", round(max(va_acc), 4))

  1轮：训练0.7250，验证 0.7083
  5轮：训练0.8000，验证 0.8067
 10轮：训练0.7650，验证 0.7367
 50轮：训练0.8250，验证 0.7900
100轮：训练0.8300，验证 0.7550
200轮：训练0.8650，验证 0.7383
500轮：训练0.9050，验证 0.7300
验证准确率最高：第 16 轮 0.815


训练准确率整体上升（1 轮 0.725 → 500 轮 0.905），模型越来越会“背”训练样本；验证集：5 轮 0.8067 很高，10 轮就掉下去，16 轮到顶峰 0.815，之后一路下滑，500 轮只剩 0.73。

说明：超过 16 轮之后，增加弱分类器对新数据反而有害，过拟合。

所以，这就是**早停（early stopping）**思想：我们可以监控验证集效果，一旦验证集开始持续下降，就停止训练，不要继续加轮数了。


## 9. 信贷数据上的 AdaBoost

最后在 `data/信贷数据.csv` 上跑一遍：

- 数值特征：年龄、月收入、负债收入比、额度使用率、逾期次数、工作年限，直接使用；
- 类别特征：学历、住房、职业，做 One-Hot 编码；
- 按 `是否违约` 分层抽样，20% 作测试集；
- 弱分类器仍是树桩，`n_estimators=300`。

用 `staged_predict_proba` 输出每一轮的违约概率，再算 `roc_auc_score`。

| 轮数 | 测试集 AUC |
| --- | --- |
| 1 | 0.5000 |
| 10 | 0.7266 |
| 50 | 0.7400 |
| 100 | 0.7543 |
| 300 | 0.7537 |

1 轮时 AUC 为 0.5，相当于随机猜；10 轮后升到 0.7266，之后提升有限，100 轮附近达到 0.7543，300 轮反而略降到 0.7537——同样说明轮数不是越多越好。

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier

# 加载数据
df = pd.read_csv('../data/信贷数据.csv')

# 分开定义：数值型特征、类别型特征
num = ["年龄", "月收入", "负债收入比", "额度使用率", "逾期次数", "工作年限"]
cat = ["学历", "住房", "职业"]

# 划分训练集、测试集，测试集占20%
X_train, X_test, y_train, y_test = train_test_split(
    df[num+cat], df['是否违约'].to_numpy(), 
    test_size=0.2, stratify=df['是否违约'], random_state=42
)

# 预处理流水线：数值列原样保留（passthrough）；分类列做OneHot独热编码
enc = ColumnTransformer([
    ('num', "passthrough", num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
]).fit(X_train)

# 对训练、测试数据执行编码转换
E_train, E_test = enc.transform(X_train), enc.transform(X_test)

# 构建AdaBoost：弱学习器是深度=1的决策树（树桩，只能切一刀），最多300轮
ada_c = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=300,
    random_state=0
).fit(E_train, y_train)

# staged_predict_proba：依次返回【前1个、前2个……前300个弱分类器组合】的预测概率
# roc_auc_score 提取每个阶段的正类概率 p[:,1] 计算 AUC
auc = [roc_auc_score(y_test, p[:, 1]) for p in ada_c.staged_predict_proba(E_test)]

# 挑选几个关键轮数打印结果：1、10、50、100、300轮，保留4位小数
print("测试集 AUC:", {m: round(auc[m-1], 4) for m in [1, 10, 50, 100, 300]})

测试集 AUC: {1: 0.5, 10: 0.7266, 50: 0.74, 100: 0.7543, 300: 0.7537}
